# Notebook for Library merging

In [2]:
# Auto-reload edited modules (Benchmarking/, Models/, ...) before each cell runs,
# so editing a .py file takes effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

In [3]:
# Imports
from sklearn.model_selection import ParameterGrid
from Models.config_parser import load_config, load_dataset_config, load_seed
from Models.dataset_and_models import Dataset, Model
import pandas as pd

In [4]:
# Load configurations
CONFIG = "../configs/config.yaml"

model_config   = load_config(CONFIG)        # {model_key: {param: [values]}}
dataset_config = load_dataset_config(CONFIG) # {dataset_key: {n_features: [...], n_samples: [...]}}
seed = load_seed(CONFIG)

In [5]:
# All model × hyperparameter combinations
model_runs = [
    (model_key, params)
    for model_key, param_grid in model_config.items()
    for params in ParameterGrid(param_grid)
]

# All dataset × (n_features, n_samples) combinations
dataset_runs = [
    (dataset_key, params)
    for dataset_key, param_grid in dataset_config.items()
    for params in ParameterGrid(param_grid)
]


In [6]:
import yaml
from Benchmarking.backends import (
    ShapApproxBackend,
    ShapIQApproxBackend,
    LightShapApproxBackend,
    DalexApproxBackend,
)

# Each (model, dataset) cell is one runner.run() call. Within a cell the runner executes
# one exact oracle plus every selected approximation spec (library × approximator × budget),
# writing one results.csv row per backend run. So the real total — the number you can
# compare against len(results) in the reader cell below — multiplies the cells by the
# per-cell backend count, not just the cells themselves.
n_cells = len(model_runs) * len(dataset_runs)

with open(CONFIG) as f:
    _bench = yaml.safe_load(f)["benchmark"]

# A library only runs the approximator slots it actually implements (a backend may
# declare SUPPORTED_APPROXIMATORS to opt out — e.g. dalex has no kernel variant), so the
# spec count is the sum over libraries of their supported slots, not a flat product.
_backend_by_lib = {
    "shap": ShapApproxBackend,
    "shapiq": ShapIQApproxBackend,
    "lightshap": LightShapApproxBackend,
    "dalex": DalexApproxBackend,
}
n_specs = len(_bench["budgets"]) * sum(
    sum(1 for appr in _bench["approximators"]
        if appr in getattr(_backend_by_lib[lib], "SUPPORTED_APPROXIMATORS", _bench["approximators"]))
    for lib in _bench["libraries"]
)
runs_per_cell = 1 + n_specs  # 1 exact oracle + approximation sweep
total_runs = n_cells * runs_per_cell

print(f"{len(model_runs)} model configs × {len(dataset_runs)} dataset configs = {n_cells} (model, dataset) cells")
print(
    f"per cell: 1 oracle + {n_specs} approx specs "
    f"({len(_bench['libraries'])} libs × {len(_bench['approximators'])} approximators "
    f"× {len(_bench['budgets'])} budgets, minus slots a library doesn't implement) "
    f"= {runs_per_cell} backend runs"
)
print(f"→ {n_cells} × {runs_per_cell} = {total_runs} total backend runs / results.csv rows")

4 model configs × 9 dataset configs = 36 (model, dataset) cells
per cell: 1 oracle + 7 approx specs (4 libs × 2 approximators × 1 budgets, minus slots a library doesn't implement) = 8 backend runs
→ 36 × 8 = 288 total backend runs / results.csv rows


### Iterate trough every combination and train the model for explanation

In [ ]:
import warnings
import yaml
from tqdm import tqdm
from Benchmarking import BenchmarkRunner
from Benchmarking.backends import (
    ShapTrueValueBackend,
    ShapApproxBackend,
    ShapIQApproxBackend,
    LightShapApproxBackend,
    DalexApproxBackend,
)

warnings.filterwarnings(
    "ignore",
    message="Not all budget is required due to the border-trick.",
    category=UserWarning,
    module="shapiq",
)
warnings.filterwarnings(
    "ignore",
    message="The sample size is larger than the number of data points in the background set.*",
    category=UserWarning,
    module="shapiq",
)

# Map config library names to their approximation backend classes.
APPROX_BACKEND_BY_LIBRARY = {
    "shap": ShapApproxBackend,
    "shapiq": ShapIQApproxBackend,
    "lightshap": LightShapApproxBackend,
    "dalex": DalexApproxBackend,
}

# All benchmark knobs come from the config's `benchmark` section (loaded the same
# way as the model/dataset configs). No defaults: every knob must be set there.
with open(CONFIG) as f:
    benchmark_config = yaml.safe_load(f)["benchmark"]
n_background = benchmark_config["n_background"]
n_eval = benchmark_config["n_eval"]
seed = benchmark_config["seed"]
imputer = benchmark_config["imputer"]
BUDGETS = benchmark_config["budgets"]
APPROXIMATORS = benchmark_config["approximators"]
APPROX_BACKENDS = [APPROX_BACKEND_BY_LIBRARY[library] for library in benchmark_config["libraries"]]

# Exact ground truth: shap.Explainer auto-detects TreeSHAP / LinearSHAP — polynomial,
# exact, and (with background data) interventional == the marginal value function that
# every approximator below targets. One cheap oracle per cell, recomputed each run.
TRUE_VALUE_BACKENDS = [ShapTrueValueBackend]

# Approximation sweep: only the (library × approximator × budget) combinations
# selected in the config. The nominal budget differs in units across libraries, so
# n_model_evals — the real model-call count recorded by the runner — is the
# comparable budget axis. A backend may declare SUPPORTED_APPROXIMATORS to opt out of
# slots it has no method for (dalex has only ordering-sampling SHAP, no kernel variant),
# so those pairs are skipped rather than run as silent duplicates.
approximation_specs = [
    (backend, {"approximator": approximator, "budget": budget})
    for backend in APPROX_BACKENDS
    for approximator in APPROXIMATORS
    for budget in BUDGETS
    if approximator in getattr(backend, "SUPPORTED_APPROXIMATORS", APPROXIMATORS)
]

runner = BenchmarkRunner(
    true_value_backends=TRUE_VALUE_BACKENDS,
    approximation_specs=approximation_specs,
    output_csv="../Benchmarking/results.csv",
    n_background=n_background,
    n_eval=n_eval,
    seed=seed,
    imputer=imputer,
)

total_weight = sum(
    dataset_params.get("n_features", 1)
    for _, dataset_params in dataset_runs
    for _ in model_runs
)

with tqdm(total=total_weight, desc="benchmark", unit="feat") as bar:
    for dataset_key, dataset_params in dataset_runs:
        dataset_enum = Dataset[dataset_key.upper()]
        ds = dataset_enum.load_dataset(**dataset_params, seed=seed)
        X, y = ds["X"], ds["y"]
        weight = dataset_params.get("n_features", 1)

        for model_key, model_params in model_runs:
            bar.set_postfix_str(f"{dataset_key} nf={dataset_params.get('n_features')} | {model_key}")
            trainer = Model[model_key.upper()].get_model_with_params(dataset_enum, model_params, seed=seed)
            trainer.fit(X, y, task=ds["task"])

            runner.run(
                model=trainer.get_model(),
                X=X,
                run_meta={
                    "dataset": dataset_key,
                    "model": model_key,
                    "n_features": dataset_params.get("n_features"),
                    "n_samples": dataset_params.get("n_samples"),
                },
            )
            bar.update(weight)

print("Done. Results written to ../Benchmarking/results.csv")

benchmark:   0%|          | 0/1604 [00:00<?, ?feat/s, ames_housing nf=4 | random_forest]Background dataset has 200 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=200 when initializing the masker.
Using 200 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


In [ ]:
RESULTS_CSV = "../Benchmarking/results.csv"

results = pd.read_csv(RESULTS_CSV)
n_before = len(results)

# A dalex-only append run re-emits the shap_true_value oracle row in every cell (the
# runner must recompute it to score dalex), duplicating the oracle rows already on file.
# Collapse each (cell, backend, approximator, budget) to one row — keep="last" prefers the
# most recent write. NaN approximator/budget on the true_value rows compare equal, so the
# duplicate oracle rows collapse too.
results = results.drop_duplicates(
    subset=["dataset", "model", "n_features", "n_samples",
            "backend", "approximator", "budget"],
    keep="last",
)

# Overwrite the CSV in place with the de-duplicated content.
results.to_csv(RESULTS_CSV, index=False)
print(f"de-duplicated {n_before} -> {len(results)} rows; overwrote {RESULTS_CSV}")

In [ ]:
print(f"{len(results)} rows written")

cols = [
    "dataset", "model", "n_features", "backend", "approximator", "budget",
    "n_model_evals", "runtime_s", "mean_abs_diff", "relative_mae",
    "sign_agreement", "mean_sample_rho",
]
results[cols].head(20)

results

## Ground-truth validation (run once)

The exact reference used across the whole sweep is `shap`'s auto-detected **polynomial**
explainer (TreeSHAP / LinearSHAP) — fast at any feature count. To confirm this cheap oracle
really equals the exact Shapley value, compare it on a small problem (california-housing,
8 features) against an **independent brute-force exact**: `shapiq`'s general model-agnostic
explainer at the maximum budget `2⁸ = 256` (full coalition enumeration) — a different
library *and* a different algorithm. Agreement to numerical precision validates the oracle.
One-off check, **not** part of the main sweep.

In [ ]:
from Benchmarking.backends import ShapTrueValueBackend, ShapIQTrueValueBackend

val_ds = Dataset.CALIFORNIA_HOUSING.load_dataset(n_samples=300, n_features=8, seed=seed)
Xv, yv = val_ds["X"], val_ds["y"]
val_trainer = Model.RANDOM_FOREST.get_model_with_params(
    Dataset.CALIFORNIA_HOUSING, {"n_estimators": 50, "max_depth": 6}, seed=seed
)
val_trainer.fit(Xv, yv, task=val_ds["task"])
val_model = val_trainer.get_model()

bg, xe = Xv.iloc[:100], Xv.iloc[100:120]

# Oracle: polynomial TreeSHAP (the reference used across the whole sweep)
poly = ShapTrueValueBackend(val_model, bg).run_explainer(xe)

# Independent brute-force exact: shapiq's general model-agnostic explainer at the maximum
# budget = 2^n_features (full coalition enumeration) — different library, different algorithm.
shapiq_exact = ShapIQTrueValueBackend(val_model, bg).run_explainer(xe)

print("max |TreeSHAP - shapiq exact (full budget)|:", float((poly - shapiq_exact).abs().values.max()))
print("-> agree to numerical precision; oracle validated.")